---

# 🧠 BERT 深度学习笔记 (核心精华版)

## 一、 核心概述 (Core Overview)
* **全称**：**B**idirectional **E**ncoder **R**epresentations from **T**ransformers
* **诞生背景**：Google 于 2018 年提出。它的出现彻底改变了 NLP（自然语言处理）的游戏规则，将 NLP 带入了“大规模预训练 + 领域微调”的新时代。
* **最大创新**：利用 Transformer 的 Encoder 结构，实现了**真正的深度双向**（Deep Bidirectional）文本表示。它不仅能看到左边的词，也能同时看到右边的词，完美融合全局上下文。

## 二、 模型结构 (Architecture)
BERT 的物理骨架其实非常单纯：它就是**堆叠的 Transformer Encoder**。
* **Transformer Encoder**：利用 Self-Attention 机制，让句子中的每个词都能与其他所有词进行交互（参考你之前看的网状连线图）。
* **两个主流版本**：
    * **BERT-base**：12 层 Encoder (Layers)，隐藏层维度 768 (Hidden Size)，12 个注意力头 (Attention Heads)，总参数量约 1.1 亿 (110M)。
    * **BERT-large**：24 层 Encoder，隐藏层维度 1024，16 个注意力头，总参数量约 3.4 亿 (340M)。

## 三、 输入表示 (Input Representation) —— BERT 的“消化系统”
BERT 接收的输入不是单一的词向量，而是**三种 Embedding 的叠加**。这是一个极其重要的考点。对于任何一个输入的 Token，它的最终向量表示公式为：
$$Embedding = E_{token} + E_{segment} + E_{position}$$

1.  **Token Embedding (词向量)**：
    * 采用 **WordPiece** 分词算法，把常见的词保留，罕见的词拆成子词（Subword），有效处理 OOV（未登录词）问题。
    * **特殊符**：句首强行插入 `[CLS]`（用于整句分类），句尾或句间插入 `[SEP]`（用于分隔句子）。
2.  **Segment Embedding (句子向量)**：
    * 用于区分当前词属于“句子 A”还是“句子 B”。属于第一句的词加上特征 $E_A$，属于第二句的词加上特征 $E_B$。如果只有单句输入，则全是 $E_A$。
    * 会创建一个只有两行的词向量矩阵，对于第一个句子中的token，加上第一行词向量，对于第二个句子而言同理。
    * **索引 0：代表“句子 A”每个token专属特征向量。**
    * **索引 1：代表“句子 B”每个token的专属特征向量。**
3.  **Position Embedding (位置向量)**：
    * 由于 Self-Attention 本身没有顺序概念（词打乱了算出来的注意力是一样的），必须人为加入位置信息。
    * 与原版 Transformer 使用正弦/余弦函数计算固定位置编码不同，**BERT 的位置编码是依靠模型自己训练学出来的**（Learned Absolute Position Embeddings），最大支持 512 个长度。

## 四、 预训练任务 (Pre-training) —— BERT 的“灵魂”
BERT 为什么这么懂人类语言？因为它在海量无标注语料（维基百科+图书语料库，共 33 亿词）上做了两个极其精妙的“自监督填空题”：

### 任务 1：Masked Language Model (MLM, 掩码语言模型)
* **目的**：让模型具备双向理解能力（根据上下文猜词）。
* **做法**：随机将输入序列中 15% 的词“盖住”（Mask掉），然后让模型去预测这些被盖住的词是什么。
* **80-10-10 策略（细节考点）**：如果全用 `[MASK]` 符号替换，预训练阶段有 `[MASK]`，而微调阶段（真实应用中）没有 `[MASK]`，会导致数据分布不一致。因此，这 15% 被选中的词：
    * **80%** 的概率被替换为 `[MASK]`（例如：`我的狗是毛茸茸的` -> `我的狗是[MASK]茸的`）。
    * **10%** 的概率被替换为**随机词**（例如： -> `我的狗是苹果茸的`，逼迫模型不能偷懒，必须结合上下文判断这个词对不对）。
    * **10%** 的概率**保持不变**（例如： -> `我的狗是毛茸茸的`，告诉模型：有时候输入就是对的，你要保持自信）。

### 任务 2：Next Sentence Prediction (NSP, 下一句预测)
* **目的**：让模型理解句子与句子之间的逻辑关系（对问答系统、推理任务至关重要）。
* **做法**：输入句子 A 和句子 B，让模型判断 B 是不是 A 在原文中紧挨着的下一句话。
* **样本构建**：50% 的数据 A 和 B 是真实的连续句子（标签为 `IsNext`）；50% 的数据 B 是从语料库里随机抽取的句子（标签为 `NotNext`）。模型通过最前面的 `[CLS]` 向量来输出二分类结果。

## 五、 微调 (Fine-Tuning) —— 成为“全能打工人”
预训练好的 BERT 就像一个通读了人类所有书籍的大学生，你只需要给它一点点特定任务的范例（微调），它就能胜任各种工作。

* **文本分类（如情感分析）**：直接取第 12 层（最后一层）的 `[CLS]` 向量，接一个线性分类器输出概率。
* **序列标注（如命名实体识别 NER）**：不需要 `[CLS]`，直接取每个词对应的最后一层输出向量，各自接分类器，判断每个词是“人名”、“地名”还是“其他”。
* **阅读理解（如问答 SQuAD）**：输入是 [问题 + 篇章]，模型只需要输出两个指针：答案在篇章中的“起始位置”和“结束位置”。

---

### BERT 核心考点笔记 (TIPS)

* **1. 文本分类任务中取 `[CLS]` 向量的原因：**
  `[CLS]` 作为一个无实际语义的特殊符号，自身初始信息对结果干扰最小。在经过多层自注意力机制后，它能最客观、公平地融合整句话的全局上下文信息。

* **2. MLM 任务中 15% 掩码词里，有 10% 保持不变的原因：**
  为了缓解预训练阶段（输入有 `[MASK]`）和微调阶段（真实应用场景无 `[MASK]`）输入数据存在的 **分布不一致（Mismatch）** 问题。10% 保持不变是为了让模型在预训练时就“见过”微调时的样子。它建立了一座桥梁，确保模型在处理真实词汇时，也能像处理 [MASK] 一样，产出高质量的、具备上下文意识的特征向量，而不是在微调阶段面对“没有遮盖的句子”时感到水土不服。有了这 10% 的原词训练，模型才能学会在“本体语义”和“上下文语义”之间寻找平衡，从而在微调阶段做到既不盲从上下文，也不孤立看自己。


  

* **3. MLM 任务中 15% 掩码词里，有 10% 替换为随机词汇的原因：**
  为了防止模型“偷懒”。如果只提供 `[MASK]` 和原词，模型看到原词就会直接复制输出。加入随机词作为噪音，能逼迫模型不再盲目信任当前的输入，**必须始终依赖上下文来验证和纠错**，从而学到更深层的上下文关联。

* **4. 分布式表征（Contextual Representation）在 BERT 中的体现：**
  一个词的最终特征向量是**动态的**。它的表示不能仅仅取决于这个词本身（如传统的 Word2Vec），而必须高度融合它在当前句子中所处的具体语境和周围的上下文信息。
* **5. 掩码的生成机制（静态 vs 动态）**：
    原始 BERT 为了计算效率，采用静态掩码（数据复制 10 份，固定 10 种 Mask 模式）。但更优的做法是 RoBERTa 提出的动态掩码 (Dynamic Masking)，即在每次数据输入模型前动态生成 Mask。这样能确保模型在不同的 Epoch 面对同一句话时，每次都需要预测不同的词，从而最大化地榨取每一句话的上下文特征，提高数据利用率。

---

# 深度学习预训练技术笔记：BERT 的掩码独立性假设及其演进

## 1. 核心痛点：什么是“掩码独立性假设”？

BERT 的核心预训练任务是 **MLM (Masked Language Modeling)**，即随机遮掩句子中 15% 的 Token，让模型去预测它们。

在真实的自然语言中，如果句子 $C$ 中有多个词 $x_1, x_2$ 被同时遮掩，它们之间往往存在极强的语义绑定或搭配关系（例如“纽”和“约”、“苹”和“果”）。因此，它们同时出现的**真实联合概率分布**应该是：
$$P(x_1, x_2 | C)$$
这意味着预测 $x_1$ 必须高度依赖于 $x_2$。

**BERT 的缺陷：**
由于 BERT 是一次性并行输出所有被 Mask 位置的预测结果，它的网络架构在数学上做了一个强硬的**条件独立性假设**。它认为在给定上下文 $C$ 的情况下，$x_1$ 和 $x_2$ 的生成是互不干扰的：
$$P(x_1, x_2 | C) \approx P(x_1 | C) \times P(x_2 | C)$$

**真实的公式为：**
$$P(x_1, x_2 | C) = P(x_1 | C) \times P(x_2 | x_1, C)$$

**造成的致命后果：**
1. **模型“作弊”**：当“苹果”被遮掩成“苹 `[MASK]`”时，模型不需要理解整句话的上下文，只需看到“苹”字就能利用局部概率极大概率猜出“果”字。
2. **长距离依赖能力弱**：因为模型可以轻易通过这种“词汇内部的局部作弊”来降低 Loss，它就失去了去学习跨越整个句子的深层语义逻辑的动力。

---

## 2. 演进与改进方案

为了打破这种无效的独立性假设，逼迫模型学习真正的全局语义，后续的预训练模型（如 RoBERTa, SpanBERT, MacBERT 等）在构建 Mask 任务时提出了以下核心改进：

### 改进方案一：WWM (Whole Word Masking，全词掩码)
* **核心思想**：如果一个完整词汇的某一部分被选中 Mask，那么**必须把这个词的剩余部分连锅端，全部 Mask 掉**。
* **解决的痛点**：主要解决**分词算法（Tokenizer）**带来的作弊问题。
    * **中文场景**（按字切分）：强制把“苹果”变成 `[MASK] [MASK]`，而不是“苹 `[MASK]`”。
    * **英文场景**（Subword 切分）：强制把单词 “surfboarding” 拆出的 “surf”, “##board”, “##ing” 一起 Mask，防止模型通过词根词缀无脑补全。

### 改进方案二：Span Masking (连续片段掩码，以 SpanBERT 为代表)
* **核心思想**：不再随机、零散地挑选独立的 Token，而是基于几何分布，**一次性遮掩一段连续的词（Span）**（通常长度为 3~4 个 Token）。
* **解决的痛点**：彻底阻断利用短距离搭配词组作弊的路径。例如原句“他正在弹钢琴”，如果随机 Mask 可能会变成“他正在弹 `[MASK]` 琴”，模型很容易猜出。Span Masking 会将其变成“他正在 `[MASK] [MASK] [MASK]`”，迫使模型必须结合主语“他”和更广泛的语境来推断整个动作和实体。

---

## 3. 核心总结与升华（面试 / 答辩提要）

> **一句话总结**：
> 从 BERT 到 SpanBERT / WWM 的演进，本质上是**预训练任务难度（Task Difficulty）的升级**。
> 
> 原始 BERT 的独立性假设留下了一个容易被模型利用的“局部捷径”。通过引入连续片段掩码和全词掩码，我们强行掐断了这条捷径，使得模型无法再利用词汇内部的高频共现概率（联合概率）来作弊，从而倒逼 Transformer 真正发挥其自注意力机制的优势，去建立更深、更长距离的语义表征。